# LAB 03 — Delta Lake Fundamentals: Lifecycle Management

**Databricks Fundamentals | ~80 min | Difficulty ★★★★☆**

---

## Scenario

You maintain the **product catalog** for RetailHub.
New products arrive weekly, prices change, discontinued items must be removed,
and any accidental data corruption must be recoverable.

You will implement the full **Delta Lake lifecycle**:
- Create a table with constraints and a generated column
- Run DML (INSERT / UPDATE / DELETE / MERGE)
- Audit every change via `DESCRIBE HISTORY`
- Recover from a disaster using Time Travel + RESTORE
- Clean up storage with VACUUM and understand the trade-off

## Task map

| # | Title | Type | Difficulty |
|---|-------|------|-----------|
| 01 | Create table with constraints | GIVEN | ★ |
| 02 | Schema enforcement in action | GIVEN + ❓ | ★★ |
| 03 | INSERT a product batch | **TODO** | ★★ |
| 04 | UPDATE: price change + restock | **TODO** | ★★ |
| 05 | DELETE: retire discontinued products | **TODO** | ★★ |
| 06 | DESCRIBE DETAIL + HISTORY audit | **TODO** | ★★★ |
| 07 | MERGE INTO — weekly delta upsert | **TODO** | ★★★★ |
| 08 | Schema evolution: add `supplier` column | **TODO** | ★★★ |
| 09 | Time Travel — VERSION AS OF | **TODO** | ★★★ |
| 10 | Time Travel — TIMESTAMP AS OF + diff | **TODO** | ★★★ |
| 11 | Simulate disaster → RESTORE | **TODO** | ★★★★ |
| 12 | VACUUM — cleanup & impact on Time Travel | **TODO** | ★★★★ |

In [ ]:
# Setup — run first
%run ../setup/00_setup

TABLE = f"{CATALOG}.silver.lab_products"
print(f"Working table: {TABLE}")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
# Task 01 — Create product catalog table with constraints           [GIVEN]
# ─────────────────────────────────────────────────────────────────────────
# Delta supports:
#   NOT NULL       — enforced on every write
#   CHECK(expr)    — custom row-level predicate
#   DEFAULT value  — applied when column is omitted in INSERT
#   GENERATED AS   — computed from other columns, NEVER inserted manually

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.silver")
spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

spark.sql(f"""
    CREATE TABLE {TABLE} (
        product_id    STRING  NOT NULL,
        name          STRING  NOT NULL,
        category      STRING,
        unit_price    DOUBLE  NOT NULL,
        stock_qty     INT     DEFAULT 0,
        is_active     BOOLEAN DEFAULT TRUE,
        created_date  DATE,
        price_band    STRING  GENERATED ALWAYS AS (
                          CASE
                              WHEN unit_price < 10  THEN 'budget'
                              WHEN unit_price < 50  THEN 'mid'
                              WHEN unit_price < 100 THEN 'premium'
                              ELSE 'luxury'
                          END
                      ),
        CONSTRAINT chk_price CHECK (unit_price > 0),
        CONSTRAINT chk_stock CHECK (stock_qty  >= 0)
    )
    USING DELTA
""")

spark.sql(f"""
    INSERT INTO {TABLE}
        (product_id, name, category, unit_price, stock_qty, is_active, created_date)
    VALUES
        ('P001', 'Laptop Pro 15',    'Electronics', 1299.99, 50, TRUE, '2024-01-05'),
        ('P002', 'Wireless Mouse',   'Electronics',   29.99,200, TRUE, '2024-01-05'),
        ('P003', 'Office Chair',     'Furniture',    349.00, 30, TRUE, '2024-01-10'),
        ('P004', 'USB-C Hub 7-in-1', 'Electronics',   49.99,150, TRUE, '2024-01-15'),
        ('P005', 'Standing Desk',    'Furniture',    599.00, 20, TRUE, '2024-02-01')
""")

spark.table(TABLE).show(truncate=False)
print("✅ Table created — this is version 0")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
    # Task 02 — Schema enforcement in action                       [GIVEN + ❓]
    # ─────────────────────────────────────────────────────────────────────────
    # Delta REJECTS writes that violate schema or constraints.

    from datetime import date
    from pyspark.sql import Row
    from pyspark.sql.types import (
        StructType, StructField,
        StringType, DoubleType, IntegerType, BooleanType, DateType
    )

    # Attempt A — numeric column supplied as STRING
    print("=== Attempt A: unit_price as STRING (wrong type) ===")
    try:
        bad_schema = StructType([
            StructField("product_id",   StringType(),  False),
            StructField("name",         StringType(),  False),
            StructField("category",     StringType(),  True),
            StructField("unit_price",   StringType(),  False),   # ← deliberately wrong
            StructField("stock_qty",    IntegerType(), True),
            StructField("is_active",    BooleanType(), True),
            StructField("created_date", DateType(),    True),
        ])
        bad_df = spark.createDataFrame(
            [Row("P099", "Bad Product", "Test", "NOT_A_NUMBER", 10, True, date(2024, 6, 1))],
            bad_schema
        )
        bad_df.write.format("delta").mode("append").saveAsTable(TABLE)
        print("  Written — unexpected success!")
    except Exception as e:
        print(f"  ❌ Caught {type(e).__name__}: {str(e)[:200]}")

    # Attempt B — CHECK constraint violation (negative price)
    print("
=== Attempt B: negative unit_price (CHECK constraint) ===")
    try:
        spark.sql(f"""
            INSERT INTO {TABLE} (product_id, name, unit_price, created_date)
            VALUES ('P099', 'Ghost', -5.0, '2024-06-01')
        """)
        print("  Written — unexpected success!")
    except Exception as e:
        print(f"  ❌ Caught {type(e).__name__}: {str(e)[:200]}")

    # ❓ Which layer (Spark engine vs Delta) rejects Attempt A vs Attempt B?
    # ❓ How is a GENERATED column different from a regular withColumn()?
    # ❓ What happens to price_band when you UPDATE unit_price?

### Task 03 — INSERT a new product batch  `[TODO]`

A new product drop arrives. Insert 3 new rows — this creates **version 2**.

> **Important:** do NOT supply `price_band` — it is `GENERATED ALWAYS AS`
> and is computed automatically.

New products:

| product_id | name | category | unit_price | stock_qty | is_active | created_date |
|---|---|---|---|---|---|---|
| P006 | Noise-Cancelling Headphones | Electronics | 199.99 | 75 | TRUE | 2024-03-01 |
| P007 | Ergonomic Keyboard | Electronics | 89.00 | 120 | TRUE | 2024-03-01 |
| P008 | Monitor 27" 4K | Electronics | 449.00 | 40 | TRUE | 2024-03-05 |

**API hint:**
```sql
INSERT INTO catalog.schema.table
    (product_id, name, category, unit_price, stock_qty, is_active, created_date)
VALUES
    (...), (...), (...)
```

In [ ]:
# Task 03 — TODO: insert 3 new products

spark.sql(f"""
    -- TODO: INSERT INTO {TABLE} (...) VALUES (...), (...), (...)
""")

print(f"Row count after INSERT: {spark.table(TABLE).count()}")

# Verify generated column was assigned automatically
spark.table(TABLE).select("product_id","name","unit_price","price_band").show(truncate=False)
# ❓ What price_band values were auto-assigned to the new products?

### Task 04 — UPDATE: price change + restock  `[TODO]`

Write **two separate UPDATE statements** (each creates a new Delta version):

- **Event A:** `Laptop Pro 15` (P001) price drops **10%**
- **Event B:** `Standing Desk` (P005) restocked by **+50 units**

**API hint:**
```sql
UPDATE catalog.schema.table
SET    column = expression
WHERE  condition

-- expressions:
unit_price = unit_price * 0.90
stock_qty  = stock_qty + 50
```

> After updating P001's price, inspect `price_band` — did it auto-update? Why?

In [ ]:
# Task 04 — TODO: two UPDATE statements

# Event A — 10% price drop on P001
spark.sql(f"""
    -- TODO: UPDATE {TABLE} SET unit_price = unit_price * 0.90 WHERE product_id = 'P001'
""")

# Event B — restock P005
spark.sql(f"""
    -- TODO: UPDATE {TABLE} SET stock_qty = stock_qty + 50 WHERE product_id = 'P005'
""")

spark.table(TABLE).select("product_id","name","unit_price","stock_qty","price_band")                       .show(truncate=False)
# ❓ Did price_band change for P001 after the price drop?

### Task 05 — DELETE: retire discontinued products  `[TODO]`

**Step 5a (given):** Mark P004 and P007 as retired — run as-is.

**Step 5b (TODO):** Delete all rows where `is_active = FALSE AND stock_qty = 0`.

**API hint:**
```sql
DELETE FROM catalog.schema.table
WHERE  condition_a
AND    condition_b
```

> Why two steps? The `is_active` flag is set by a business process (possibly days
> before the cleanup job runs). Separating the flag from DELETE keeps both events
> in the audit log independently.

In [ ]:
# Task 05a — mark P004 and P007 as retired (GIVEN — run as-is)
    spark.sql(f"UPDATE {TABLE} SET is_active = FALSE, stock_qty = 0 WHERE product_id IN ('P004','P007')")
    print("After marking retired:")
    spark.table(TABLE).select("product_id","name","is_active","stock_qty").show(truncate=False)

    # Task 05b — TODO: DELETE retired zero-stock products
    spark.sql(f"""
        -- TODO: DELETE FROM {TABLE} WHERE is_active = FALSE AND stock_qty = 0
    """)

    print(f"
Row count after DELETE: {spark.table(TABLE).count()}")
    spark.table(TABLE).show(truncate=False)

### Task 06 — Audit: DESCRIBE DETAIL + DESCRIBE HISTORY  `[TODO]`

Delta records **every write** in the transaction log (`_delta_log/`).

**API hints:**
```python
# Physical file metadata (1 row):
spark.sql("DESCRIBE DETAIL catalog.schema.table")
# Key fields: numFiles, sizeInBytes, partitionColumns

# Full operation log:
spark.sql("DESCRIBE HISTORY catalog.schema.table")
# Key columns: version, timestamp, operation, operationParameters, operationMetrics
```

Your tasks:
1. Show `numFiles`, `sizeInBytes`, `partitionColumns` from DESCRIBE DETAIL
2. Show the full history: `version`, `timestamp`, `operation`, `operationMetrics`
3. From `operationMetrics` of the DELETE operation, find `numDeletedRows`

In [ ]:
# Task 06 — TODO: audit queries

    # 6a — physical metadata
    print("=== DESCRIBE DETAIL ===")
    # TODO: spark.sql(f"DESCRIBE DETAIL {TABLE}")     #           .select("numFiles","sizeInBytes","partitionColumns").show(truncate=False)

    # 6b — full history
    print("
=== DESCRIBE HISTORY ===")
    # TODO: spark.sql(f"DESCRIBE HISTORY {TABLE}")     #           .select("version","timestamp","operation","operationMetrics").show(truncate=False)

    # ❓ How many Delta versions exist right now?
    # ❓ What is numDeletedRows in the DELETE operation metrics?
    # ❓ Why might numFiles be larger than the number of active rows?

### Task 07 — MERGE INTO: weekly delta upsert  `[TODO]`

Every Monday a *weekly delta file* arrives with updates and new rows.

**Full MERGE INTO syntax:**
```sql
MERGE INTO target AS tgt
USING source AS src
ON    tgt.product_id = src.product_id

WHEN MATCHED THEN UPDATE SET
     tgt.unit_price = src.unit_price,
     tgt.stock_qty  = src.stock_qty

WHEN NOT MATCHED THEN INSERT
     (product_id, name, category, unit_price, stock_qty, is_active, created_date)
VALUES
     (src.product_id, src.name, src.category, src.unit_price,
      src.stock_qty, src.is_active, src.created_date)
```

> **Note:** omit `price_band` from both SET and INSERT VALUES — it is generated.

Source arriving this week:

| product_id | change |
|---|---|
| P001 | price → 1099.99, stock → 55 |
| P003 | price → 329.00, stock → 35 |
| P009 | Smart Watch Series 3 (NEW) 249.99 qty 80 |
| P010 | Laptop Stand Alu (NEW) 39.99 qty 200 |

In [ ]:
# Task 07 — TODO: MERGE INTO

from datetime import date
from pyspark.sql import Row

weekly_src = spark.createDataFrame([
    Row(product_id="P001", name="Laptop Pro 15",          category="Electronics",
        unit_price=1099.99, stock_qty=55,  is_active=True, created_date=date(2024,1,5)),
    Row(product_id="P003", name="Office Chair",           category="Furniture",
        unit_price=329.00,  stock_qty=35,  is_active=True, created_date=date(2024,1,10)),
    Row(product_id="P009", name="Smart Watch Series 3",   category="Wearables",
        unit_price=249.99,  stock_qty=80,  is_active=True, created_date=date(2024,4,1)),
    Row(product_id="P010", name="Laptop Stand Aluminium", category="Accessories",
        unit_price=39.99,   stock_qty=200, is_active=True, created_date=date(2024,4,1)),
])
weekly_src.createOrReplaceTempView("weekly_src")

spark.sql(f"""
    -- TODO: MERGE INTO {TABLE} AS tgt
    --       USING weekly_src AS src
    --       ON    tgt.product_id = src.product_id
    --       WHEN MATCHED     THEN UPDATE SET ...
    --       WHEN NOT MATCHED THEN INSERT (...) VALUES (...)
""")

print("After MERGE:")
spark.table(TABLE).orderBy("product_id").show(truncate=False)

# Check metrics
spark.sql(f"DESCRIBE HISTORY {TABLE} LIMIT 1")          .select("version","operation","operationMetrics").show(truncate=False)
# ❓ How many rows were updated? How many were inserted?

### Task 08 — Schema evolution: add `supplier` column  `[TODO]`

The procurement team needs a `supplier` column (STRING, nullable).
Existing rows will have `supplier = NULL` after the DDL.

**Two-step production pattern:**

```sql
-- Step 1: safe DDL (zero downtime)
ALTER TABLE catalog.schema.table ADD COLUMN supplier STRING

-- Step 2: backfill known mappings
UPDATE catalog.schema.table
SET    supplier = 'TechDistrib AG'
WHERE  category = 'Electronics'
```

Backfill **at least two** categories.

In [ ]:
# Task 08 — TODO: add supplier column + backfill

# Step 1 — ADD COLUMN
spark.sql(f"""
    -- TODO: ALTER TABLE {TABLE} ADD COLUMN supplier STRING
""")

print("Schema after ALTER:")
spark.table(TABLE).printSchema()

# Step 2 — backfill Electronics
spark.sql(f"""
    -- TODO: UPDATE {TABLE} SET supplier = 'TechDistrib AG' WHERE category = 'Electronics'
""")

# TODO: backfill at least one more category
spark.sql(f"""
    -- TODO: UPDATE {TABLE} SET supplier = '<name>' WHERE category = '<cat>'
""")

spark.table(TABLE).select("product_id","name","category","supplier").show(truncate=False)

### Task 09 — Time Travel: VERSION AS OF  `[TODO]`

**Three equivalent ways to read a past version:**
```python
# Python API
df_v0 = (spark.read
              .format("delta")
              .option("versionAsOf", 0)
              .table("catalog.schema.table"))

# SQL standard
df_v0 = spark.sql("SELECT * FROM catalog.schema.table VERSION AS OF 0")

# Databricks shorthand
df_v0 = spark.sql("SELECT * FROM catalog.schema.table@v0")
```

Tasks:
1. Read **version 0** (initial seed data) into `df_v0`
2. Find products in v0 that no longer exist in the current table  
   *(hint: `left_anti` join on `product_id`)*
3. Find products whose `unit_price` **changed** between v0 and now  
   *(hint: inner join + filter where prices differ)*

In [ ]:
# Task 09 — TODO: time travel VERSION AS OF

    from pyspark.sql import functions as F

    # 9a — TODO: read version 0
    # df_v0 = (spark.read
    #               .format("delta")
    #               .option("versionAsOf", 0)
    #               .table(TABLE))
    df_v0 = None  # replace

    df_current = spark.table(TABLE)

    # 9b — TODO: deleted products (in v0 but NOT current)
    print("Products deleted since version 0:")
    # df_deleted = df_v0.join(df_current.select("product_id"),
    #                         on="product_id", how="left_anti")
    # df_deleted.select("product_id","name").show(truncate=False)

    # 9c — TODO: products with changed unit_price
    print("
Products with changed unit_price:")
    # df_v0_p = df_v0.select("product_id", F.col("unit_price").alias("price_v0"))
    # df_cur_p = df_current.select("product_id", F.col("unit_price").alias("price_now"))
    # df_changed = df_v0_p.join(df_cur_p,"product_id")     #                     .filter(F.col("price_v0") != F.col("price_now"))
    # df_changed.show(truncate=False)

### Task 10 — Time Travel: TIMESTAMP AS OF + diff  `[TODO]`

`TIMESTAMP AS OF` is used when you know the **time window**
but not the version number.

**API hint:**
```python
history = spark.sql("DESCRIBE HISTORY catalog.schema.table")
v1_ts = history.filter("version = 1").select("timestamp").collect()[0][0]

df_at_ts = (spark.read
                 .format("delta")
                 .option("timestampAsOf", str(v1_ts))
                 .table("catalog.schema.table"))
```

Tasks:
1. Get the timestamp of **version 1** from history
2. Read the table at that timestamp into `df_at_v1`
3. Compare row counts (v1 vs current)
4. Compare column lists — notice the schema difference
5. Does v1 have the `supplier` column? Why/why not?

In [ ]:
# Task 10 — TODO: timestamp-based time travel + schema diff

# 10a — TODO: get version 1 timestamp
history = spark.sql(f"DESCRIBE HISTORY {TABLE}")
# v1_ts = history.filter("version = 1").select("timestamp").collect()[0]["timestamp"]
v1_ts = None  # replace
print(f"Version 1 timestamp: {v1_ts}")

# 10b — TODO: read at v1_ts
# df_at_v1 = (spark.read
#                  .format("delta")
#                  .option("timestampAsOf", str(v1_ts))
#                  .table(TABLE))
df_at_v1 = None  # replace

# 10c — compare row counts
if df_at_v1 is not None:
    print(f"Version 1 rows:  {df_at_v1.count()}")
print(f"Current rows:    {spark.table(TABLE).count()}")

# 10d — compare schema
if df_at_v1 is not None:
    print(f"Version 1 columns: {df_at_v1.columns}")
print(f"Current columns:   {spark.table(TABLE).columns}")

# ❓ Does v1 have the 'supplier' column?
# ❓ What does this tell you about schema evolution + time travel?
# ❓ How can this break downstream pipelines reading historical snapshots?

### Task 11 — Simulate disaster → RESTORE  `[TODO]`

A junior engineer accidentally deletes all `Electronics` products.
Detect the damage and roll the table back.

**RESTORE TABLE syntax:**
```sql
-- by version (deterministic — preferred):
RESTORE TABLE catalog.schema.table TO VERSION AS OF <n>

-- by timestamp:
RESTORE TABLE catalog.schema.table TO TIMESTAMP AS OF '<ts>'
```

**Key properties:**
- Creates a **new** Delta version — history is **never** rewritten
- The bad version remains in the log (immutable audit trail)
- Schema must be compatible

Steps:
1. Run the accidental DELETE (given)
2. Inspect DESCRIBE HISTORY to find the last-good version
3. RESTORE to that version
4. Verify the data is back
5. Re-check history — notice the new `RESTORE` entry

In [ ]:
# Task 11 — RESTORE from accidental DELETE

    # Step 1 — accidental DELETE (GIVEN — run as-is)
    print("=== BEFORE accidental DELETE ===")
    print(f"Electronics count: {spark.table(TABLE).filter("category = 'Electronics'").count()}")

    spark.sql(f"DELETE FROM {TABLE} WHERE category = 'Electronics'")

    print("
=== AFTER accidental DELETE ===")
    print(f"Total rows remaining: {spark.table(TABLE).count()}")
    spark.table(TABLE).show(truncate=False)

    # Step 2 — TODO: inspect history, identify last good version
    print("
=== DESCRIBE HISTORY ===")
    # TODO: spark.sql(f"DESCRIBE HISTORY {TABLE}")     #           .select("version","operation","timestamp").show(truncate=False)
    last_good_version = None  # TODO: set the correct integer

    # Step 3 — TODO: RESTORE
    if last_good_version is not None:
        spark.sql(f"RESTORE TABLE {TABLE} TO VERSION AS OF {last_good_version}")

    # Step 4 — verify
    print("
=== After RESTORE ===")
    print(f"Total rows: {spark.table(TABLE).count()}")
    spark.table(TABLE).show(truncate=False)

    # Step 5 — history now has a RESTORE entry
    spark.sql(f"DESCRIBE HISTORY {TABLE}")          .select("version","operation").show(truncate=False)

### Task 12 — VACUUM: storage cleanup & Time Travel trade-off  `[TODO]`

Every DML creates new Parquet files. Old files are **logically deleted** but
kept on disk for time travel. VACUUM removes them permanently.

**After VACUUM you can no longer time-travel to the vacuumed versions.**

**API hint:**
```python
# production — default 7-day retention:
spark.sql("VACUUM catalog.schema.table")

# demo only — NEVER use in production:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
spark.sql("VACUUM catalog.schema.table RETAIN 0 HOURS")
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")
```

Steps:
1. Record `numFiles` before VACUUM (given)
2. Confirm time travel to version 0 works
3. VACUUM RETAIN 0 HOURS (disable guard first, re-enable after)
4. Record `numFiles` after VACUUM
5. Try time travel to version 0 — expect an error
6. Answer the three reflection questions

In [ ]:
# Task 12 — VACUUM + verification

    # 12a — numFiles before (GIVEN)
    before = spark.sql(f"DESCRIBE DETAIL {TABLE}")                   .select("numFiles","sizeInBytes").collect()[0]
    print(f"BEFORE: numFiles={before['numFiles']}, sizeInBytes={before['sizeInBytes']:,}")

    # 12b — TODO: confirm version 0 still readable
    print("
Time travel VERSION AS OF 0 BEFORE VACUUM:")
    # TODO: read version 0, print row count

    # 12c — TODO: VACUUM (disable guard → vacuum → re-enable)
    spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")
    spark.sql(f"""
        -- TODO: VACUUM {TABLE} RETAIN 0 HOURS
    """)
    spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")

    # 12d — numFiles after (GIVEN)
    after = spark.sql(f"DESCRIBE DETAIL {TABLE}")                  .select("numFiles","sizeInBytes").collect()[0]
    print(f"
AFTER:  numFiles={after['numFiles']}, sizeInBytes={after['sizeInBytes']:,}")
    print(f"Files removed: {before['numFiles'] - after['numFiles']}")

    # 12e — TODO: attempt version 0 time travel — should raise an error
    print("
Time travel VERSION AS OF 0 AFTER VACUUM (should fail):")
    try:
        # TODO: attempt read of version 0 here
        pass
    except Exception as e:
        print(f"  ❌ {type(e).__name__}: {str(e)[:240]}")

    # ❓ What is the default VACUUM retention period and WHY is it 7 days?
    # ❓ DESCRIBE HISTORY still shows ALL versions after VACUUM — why?
    # ❓ In which scenario would you lower retention below 7 days?